## Module 6-4o Explainable AI: Using `SHAP` to Understand an XGBoost Model

We'll compute SHAP value for the exact same earnings-direction model from Section 6-2, and use the same suite of SHAP plots Parker et al. (2025) use for material misstatements: global importance, directional summary plots, dependence plots, and a single-observation waterfall explanation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
import shap

### 1. Recap

Same data pipeline

In [ ]:
df = pd.read_csv('../data/comp_sample.csv', 
                 dtype={"gvkey": str,})

df = df[
    (df['indfmt'] == 'INDL') &
    (df['curcd'] == 'USD') &
    (df['costat'] == 'A')
].drop_duplicates(subset = ['gvkey', 'fyear']).copy()
df = df.sort_values(['gvkey', 'fyear']).reset_index(drop=True)

df['eps'] = (df['ni'] / df['csho']).replace([np.inf, -np.inf], np.nan) # Here we should use IBES Actual EPS. Just a simplified demo for this workshop
df['d_eps'] = df.groupby('gvkey')['eps'].diff()
df['d_eps'] = df['d_eps'].where(df.groupby('gvkey')['fyear'].diff()==1, np.nan)

# Drift: trailing (up to) 4-year average change in EPS, known as of year t
df['drift'] = (
    df.groupby('gvkey')['d_eps']
      .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
)

# Next year's change in EPS, and the drift-adjusted version used for the label
df['d_eps_lead'] = df.groupby('gvkey')['d_eps'].shift(-1).where(df.groupby('gvkey')['fyear'].diff(-1)==-1, np.nan)
df['adj_d_eps_lead'] = df['d_eps_lead'] - df['drift']

df['label'] = np.where(
    df['adj_d_eps_lead'].isna(), np.nan, (df['adj_d_eps_lead'] > 0).astype(np.int8)
)

ID_COLS = ['gvkey', 'datadate', 'fyear']
LABEL_COLS = ['eps', 'd_eps', 'drift', 'd_eps_lead', 'adj_d_eps_lead', 'label']
NON_FS_COLS = ['au']  # e.g. auditor code: an identifier, not a financial-statement item

# Items that should NOT be scaled by total assets:
NO_SCALE_COLS = ['at', 'csho', 'prcc_f']

exclude_cols = set(ID_COLS + LABEL_COLS + NON_FS_COLS)
numeric_cols = df.select_dtypes(include='number').columns
predictor_base_cols = [c for c in numeric_cols if c not in exclude_cols]

def build_features(data: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Current value, lagged value, and percentage change for each base column."""
    at_cur = data['at']
    at_lag = data.groupby('gvkey')['at'].shift(1).where(data.groupby('gvkey')['fyear'].diff()==1)

    feature_cols = {}
    for col in base_cols:
        cur = data[col]
        lag = data.groupby('gvkey')[col].shift(1).where(data.groupby('gvkey')['fyear'].diff()==1)

        if col in NO_SCALE_COLS:
            cur_scaled, lag_scaled = cur, lag
        else:
            cur_scaled = (cur / at_cur).replace([np.inf, -np.inf], np.nan)
            lag_scaled = (lag / at_lag).replace([np.inf, -np.inf], np.nan)

        pct_change = ((cur - lag) / lag.abs()).replace([np.inf, -np.inf], np.nan)

        feature_cols[f'{col}_cur'] = cur_scaled
        feature_cols[f'{col}_lag'] = lag_scaled
        feature_cols[f'{col}_pctchg'] = pct_change

    return pd.DataFrame(feature_cols, index=data.index)


X_all = build_features(df, predictor_base_cols)

feature_cols = X_all.columns.tolist()

model_df = pd.concat([df[['gvkey', 'fyear', 'label']], X_all], axis=1)
model_df = model_df.dropna(subset=['label']).reset_index(drop=True)

train = model_df[model_df['fyear'] <= 2019]
val = model_df[(model_df['fyear'] > 2019) & (model_df['fyear'] <= 2021)]
test = model_df[model_df['fyear'] == 2022]

X_train, y_train = train[feature_cols], train['label']
X_val, y_val = val[feature_cols], val['label']
X_test, y_test = test[feature_cols], test['label']

xgb_tuned = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    importance_type='gain',
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

### 2. What is a SHAP value?

SHAP is grounded in cooperative game theory (Shapley 1953): treat each feature as a "player" contributing to a "payout" (the model's prediction), and fairly split that payout across the players based on their marginal contributions across all possible orderings.

For any single prediction, SHAP decomposes the gap between the model's output for that observation, $f(x)$, and the model's average output across the sample, $E[f(x)]$, into one contribution per feature:

$$f(x) - E[f(x)] = \sum_{j} \phi_j$$

where $\phi_j$ is feature $j$'s SHAP value for that observation. A **positive** $\phi_j$ pushes the prediction above the average (here: toward predicting an earnings *increase*); a **negative** $\phi_j$ pushes it below average (toward a *decrease*). The larger $|\phi_j|$, the bigger that feature's role in *this* prediction.

In [ ]:
explainer = shap.TreeExplainer(xgb_tuned)
shap_explanation = explainer(X_test)

print(type(shap_explanation))
print('SHAP values shape (observations, features):', shap_explanation.values.shape)
print('Base value E[f(x)] (log-odds):', shap_explanation.base_values[0])

### 3. Global importance

The mean absolute SHAP value $\frac{1}{n}\sum_i |\phi_{ij}|$ is the average size of feature $j$'s push on a prediction, measured in the model's log-odds output. It is on the same scale for every feature, it is computed on the held-out test set (not the training data the trees were fit on), and — unlike gain — it is the summary of a per-observation quantity we can later break open.

`shap.plots.bar` sorts features by this quantity.

In [ ]:
shap.plots.bar(shap_explanation, max_display=20)

### 4. Beeswarm plot

- **Horizontal position** is the SHAP value: how far, and in which direction, that feature pushed that firm-year's prediction.
- **Color** is the feature's (standardized) value for that firm-year.

In [ ]:
shap.plots.beeswarm(shap_explanation, max_display=20)

### 5. Dependence plots: how one feature's effect changes with its level

In [ ]:
mean_abs_shap = pd.Series(
    np.abs(shap_explanation.values).mean(axis=0), index=feature_cols
).sort_values(ascending=False)

top3 = mean_abs_shap.head(3).index.tolist()
print('Top 3 features by mean |SHAP|:', top3)

for feat in top3:
    shap.plots.scatter(shap_explanation[:, feat], color=shap_explanation)

### 6. Explaining one observation: the waterfall plot

In [ ]:
proba = xgb_tuned.predict_proba(X_test)[:, 1]
idx = int(proba.argmax())

row = test.iloc[idx]
print(f"gvkey {row['gvkey']}, fiscal year {int(row['fyear'])}")
print(f"predicted P(earnings increase) = {proba[idx]:.3f}")
print(f"actual label = {int(row['label'])}  (1 = increase, 0 = decrease)")

shap.plots.waterfall(shap_explanation[idx])